# 05. Model Tuning

## 1. Introduction

In the previous notebook, we trained and evaluated three regression models: Dummy Regressor, Linear Regression, and Random Forest Regressor.

The Random Forest model performed best among these initial models, achieving an R² score of approximately 0.93 on the test dataset.

In this notebook, we will use **hyperparameter tuning** to search for a better Random Forest configuration.

### Objectives
- Recreate the data preparation and preprocessing pipeline.
- Split the dataset into training and testing sets.
- Use RandomizedSearchCV to search for suitable hyperparameters.
- Evaluate the tuned model using MAE, RMSE, and R².
- Compare the tuned model with the original Random Forest model.
- Save the final trained pipeline for future deployment.

In [1]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
project_root = Path.cwd().parent

data_file = (
    project_root / "data" / "processed" / "pune_property_cleaned.csv"
)

df = pd.read_csv(data_file)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

display(df.head())

Dataset loaded successfully.
Dataset shape: (29680, 29)


,additionalrooms,age,amenitiesavailable,amenitiesnot,area,balconies,bathroom,bhk,carpetarea,facing,floor,locality,neworold,opensides,overlooking,ownership,possesiondate,price,pricepersquare,projectname,roadfaceing,status,totalfloor,bhk_count,property_type,floor_number,road_width_feet,property_age_years,possession_date
0,NaN,2 - 3 years,"Lift Available,Car Parking,Power Backup,Indoor...","24 X 7 Security,Children's play area,Club Hous...","1,611.00",2.00,3.00,3 BHK Apartment,"1,209.00",West,"3,",Deccan Gymkhana,New,2.00,Road View,Co-Operative Society,NaN,27000000,16759,Bhansali Prive Rio,40 feet,Ready to move,6.00,3.00,Apartment,3.00,40.00,2.50,NaN
1,NaN,0 - 1 year,"Lift Available,Car Parking,Power Backup,24 X 7...","Children's play area,Sports Facility,Landscape...","1,279.00",NaN,2.00,2 BHK Apartment,NaN,NorthEast,1,Kharadi,New,NaN,NaN,Freehold,NaN,7610050,5950,Goel Ganga Platino,NaN,Ready to move,NaN,2.00,Apartment,1.00,NaN,0.50,NaN
2,NaN,0 - 1 year,"Lift Available,Car Parking,Power Backup,24 X 7...",Sports Facility,670.00,NaN,1.00,1 BHK Apartment,NaN,NorthEast,1,Sus,New,NaN,NaN,Freehold,NaN,4100000,6119,Vilas Javdekar Yashwin,NaN,Ready to move,NaN,1.00,Apartment,1.00,NaN,0.50,NaN
3,NaN,2 - 3 years,"Lift Available,Car Parking,Power Backup,24 X 7...","Sports Facility,Landscaped Gardens",956.00,NaN,2.00,2 BHK Apartment,615.00,NorthEast,1,Bavdhan,New,NaN,NaN,Freehold,NaN,5736000,6000,Rohan Ishan,NaN,Ready to move,NaN,2.00,Apartment,1.00,NaN,2.50,NaN
4,NaN,0 - 1 year,"Lift Available,Car Parking,Power Backup,24 X 7...","Club House,Swimming Pool,Sports Facility,Joggi...","1,706.00",1.00,3.00,3 BHK Apartment,"1,195.00",West,"6,",Deccan Gymkhana,New,2.00,Road View,Freehold,NaN,29002000,17000,Swojas Aavishkar,60 feet,Ready to move,9.00,3.00,Apartment,6.00,60.00,0.50,NaN


In [3]:
features = [
    "area",
    "bhk_count",
    "property_type",
    "locality",
    "bathroom",
    "balconies",
    "floor_number",
    "totalfloor",
    "property_age_years",
    "facing",
    "ownership",
    "neworold",
    "status"
]

X = df[features].copy()
y = df["price"].copy()

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())

Feature matrix shape: (29680, 13)
Target shape: (29680,)


,area,bhk_count,property_type,locality,bathroom,balconies,floor_number,totalfloor,property_age_years,facing,ownership,neworold,status
0,"1,611.00",3.00,Apartment,Deccan Gymkhana,3.00,2.00,3.00,6.00,2.50,West,Co-Operative Society,New,Ready to move
1,"1,279.00",2.00,Apartment,Kharadi,2.00,NaN,1.00,NaN,0.50,NorthEast,Freehold,New,Ready to move
2,670.00,1.00,Apartment,Sus,1.00,NaN,1.00,NaN,0.50,NorthEast,Freehold,New,Ready to move
3,956.00,2.00,Apartment,Bavdhan,2.00,NaN,1.00,NaN,2.50,NorthEast,Freehold,New,Ready to move
4,"1,706.00",3.00,Apartment,Deccan Gymkhana,3.00,1.00,6.00,9.00,0.50,West,Freehold,New,Ready to move


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (23744, 13)
Testing features: (5936, 13)
Training target: (23744,)
Testing target: (5936,)


In [5]:
numeric_features = [
    "area",
    "bhk_count",
    "bathroom",
    "balconies",
    "floor_number",
    "totalfloor",
    "property_age_years"
]

categorical_features = [
    "property_type",
    "locality",
    "facing",
    "ownership",
    "neworold",
    "status"
]

print("Number of numerical features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

Number of numerical features: 7
Number of categorical features: 6


In [6]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [7]:
original_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=100,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

original_model.fit(X_train, y_train)

print("Original Random Forest model trained successfully.")

Original Random Forest model trained successfully.


In [8]:
param_distributions = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 15, 25, 35],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": [1.0, "sqrt", 0.8]
}

print("Hyperparameter search space created.")

Hyperparameter search space created.


In [9]:
random_search = RandomizedSearchCV(
    estimator=original_model,
    param_distributions=param_distributions,
    n_iter=10,
    scoring="neg_mean_absolute_error",
    cv=3,
    random_state=42,
    n_jobs=1,  # Disable parallel processing
    verbose=2,
    return_train_score=True,
    error_score="raise"
)

random_search.fit(X_train, y_train)

print("Hyperparameter tuning completed.")
print("\nBest parameters:")
print(random_search.best_params_)

print(
    "\nBest cross-validation MAE:",
    round(-random_search.best_score_, 2)
)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV] END model__max_depth=15, model__max_features=sqrt, model__min_samples_leaf=4, model__min_samples_split=10, model__n_estimators=100; total time=   1.2s
[CV] END model__max_depth=15, model__max_features=sqrt, model__min_samples_leaf=4, model__min_samples_split=10, model__n_estimators=100; total time=   1.1s
[CV] END model__max_depth=15, model__max_features=sqrt, model__min_samples_leaf=4, model__min_samples_split=10, model__n_estimators=100; total time=   1.0s
[CV] END model__max_depth=15, model__max_features=sqrt, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=   2.4s
[CV] END model__max_depth=15, model__max_features=sqrt, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=   2.7s
[CV] END model__max_depth=15, model__max_features=sqrt, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=   2.5s
[CV] E

In [10]:
cv_results = pd.DataFrame(random_search.cv_results_)

cv_results["Mean_CV_MAE"] = -cv_results["mean_test_score"]
cv_results["Mean_Training_MAE"] = -cv_results["mean_train_score"]

cv_results = cv_results.sort_values(
    by="Mean_CV_MAE",
    ascending=True
)

display(
    cv_results[
        [
            "params",
            "Mean_CV_MAE",
            "Mean_Training_MAE",
            "mean_fit_time"
        ]
    ].head(10).round(2)
)

,params,Mean_CV_MAE,Mean_Training_MAE,mean_fit_time
6,"{'model__n_estimators': 100, 'model__min_sampl...","2,304,209.17","1,264,495.03",12.89
3,"{'model__n_estimators': 100, 'model__min_sampl...","2,336,935.75","1,211,328.99",20.44
8,"{'model__n_estimators': 300, 'model__min_sampl...","2,448,066.45","1,712,693.96",39.37
4,"{'model__n_estimators': 100, 'model__min_sampl...","2,455,105.19","1,745,059.94",12.37
2,"{'model__n_estimators': 300, 'model__min_sampl...","2,462,368.29","1,681,181.19",34.47
9,"{'model__n_estimators': 100, 'model__min_sampl...","2,475,395.55","1,797,033.42",8.17
7,"{'model__n_estimators': 200, 'model__min_sampl...","3,312,517.69","2,808,227.31",2.54
1,"{'model__n_estimators': 100, 'model__min_sampl...","4,108,217.94","3,764,139.72",2.44
0,"{'model__n_estimators': 100, 'model__min_sampl...","4,777,790.81","4,708,121.37",1.02
5,"{'model__n_estimators': 100, 'model__min_sampl...","4,829,370.15","4,755,796.30",0.55


In [11]:
best_model = random_search.best_estimator_

print("Best tuned model selected.")
print(best_model)

Best tuned model selected.
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['area', 'bhk_count',
                                                   'bathroom', 'balconies',
                                                   'floor_number', 'totalfloor',
                                                   'property_age_years']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                

In [12]:
tuned_predictions = best_model.predict(X_test)

tuned_mae = mean_absolute_error(y_test, tuned_predictions)
tuned_rmse = np.sqrt(mean_squared_error(y_test, tuned_predictions))
tuned_r2 = r2_score(y_test, tuned_predictions)

print("Tuned Random Forest Results")
print("-" * 35)
print(f"MAE:  {tuned_mae:,.2f}")
print(f"RMSE: {tuned_rmse:,.2f}")
print(f"R²:   {tuned_r2:.4f}")

Tuned Random Forest Results
-----------------------------------
MAE:  2,132,016.27
RMSE: 4,451,221.67
R²:   0.9327


In [13]:
original_predictions = original_model.predict(X_test)

original_mae = mean_absolute_error(y_test, original_predictions)
original_rmse = np.sqrt(
    mean_squared_error(y_test, original_predictions)
)
original_r2 = r2_score(y_test, original_predictions)

comparison = pd.DataFrame(
    {
        "Model": [
            "Original Random Forest",
            "Tuned Random Forest"
        ],
        "MAE": [
            original_mae,
            tuned_mae
        ],
        "RMSE": [
            original_rmse,
            tuned_rmse
        ],
        "R2": [
            original_r2,
            tuned_r2
        ]
    }
)

display(comparison.round(4))

,Model,MAE,RMSE,R2
0,Original Random Forest,"2,063,410.31","4,407,216.77",0.93
1,Tuned Random Forest,"2,132,016.27","4,451,221.67",0.93


In [14]:
mae_improvement = (
    (original_mae - tuned_mae) / original_mae
) * 100

rmse_improvement = (
    (original_rmse - tuned_rmse) / original_rmse
) * 100

print(f"MAE improvement:  {mae_improvement:.2f}%")
print(f"RMSE improvement: {rmse_improvement:.2f}%")
print(f"Original R²:      {original_r2:.4f}")
print(f"Tuned R²:         {tuned_r2:.4f}")

MAE improvement:  -3.32%
RMSE improvement: -1.00%
Original R²:      0.9341
Tuned R²:         0.9327


In [15]:
# Select the model with the lower test MAE
if tuned_mae < original_mae:
    final_model = best_model
    final_model_name = "Tuned Random Forest"
else:
    final_model = original_model
    final_model_name = "Original Random Forest"

# Create the models directory if it does not exist
models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "final_model.joblib"

# Save the complete fitted pipeline
joblib.dump(final_model, model_path)

print("Final model selected:", final_model_name)
print("Model saved successfully at:", model_path)

Final model selected: Original Random Forest
Model saved successfully at: d:\Pune-House-Price-Prediction\models\final_model.joblib


In [16]:
loaded_model = joblib.load(model_path)

sample_features = X_test.head(5)
sample_predictions = loaded_model.predict(sample_features)

verification = pd.DataFrame(
    {
        "Actual Price": y_test.loc[sample_features.index].to_numpy(),
        "Predicted Price": sample_predictions
    }
)

verification["Absolute Error"] = (
    verification["Actual Price"] - verification["Predicted Price"]
).abs()

display(verification.round(2))

print("Saved model verification completed.")

,Actual Price,Predicted Price,Absolute Error
0,8100000,"13,250,400.00","5,150,400.00"
1,30000000,"39,803,864.00","9,803,864.00"
2,6000000,"6,027,000.00","27,000.00"
3,8500000,"10,644,500.00","2,144,500.00"
4,16000000,"9,930,963.36","6,069,036.64"


Saved model verification completed.


In [17]:
fitted_preprocessor = final_model.named_steps["preprocessor"]
fitted_forest = final_model.named_steps["model"]

transformed_feature_names = fitted_preprocessor.get_feature_names_out()

feature_importance = pd.DataFrame(
    {
        "Feature": transformed_feature_names,
        "Importance": fitted_forest.feature_importances_
    }
).sort_values(
    by="Importance",
    ascending=False
)

display(feature_importance.head(20).round(4))

,Feature,Importance
0,numeric__area,0.74
1,numeric__bhk_count,0.06
192,categorical__locality_Kharadi,0.04
32,categorical__locality_Aundh,0.01
6,numeric__property_age_years,0.01
205,categorical__locality_Koregaon Park,0.01
5,numeric__totalfloor,0.01
446,categorical__locality_Wagholi,0.01
4,numeric__floor_number,0.01
168,categorical__locality_Kalyani Nagar,0.01


In [18]:
if tuned_mae < original_mae:
    final_model = best_model
    final_model_name = "Tuned Random Forest"
else:
    final_model = original_model
    final_model_name = "Original Random Forest"

## 19. Conclusion

In this notebook, RandomizedSearchCV was used to tune the Random Forest Regressor with three-fold cross-validation.

The original Random Forest achieved a test MAE of 2,063,410.31, an RMSE of 4,407,216.77, and an R² of approximately 0.93. The tuned model achieved a test MAE of 2,132,016.27, an RMSE of 4,451,221.67, and an R² of 0.9327.

The tuned model did not improve MAE or RMSE on the held-out test set. Therefore, the original Random Forest was selected as the final model based on its lower test errors.

This result demonstrates that hyperparameter tuning does not always improve model performance. The selected pipeline will be saved and reused for deployment.

Further improvements could involve investigating prediction errors, evaluating additional features, and validating the model on representative property data.